# pycograd — compiling the DSL to Apple Silicon (Metal / MPS)

The same ambient-weights DSL as [`pycograd_demo`](pycograd_demo.ipynb) — `with params{...} as weights`, a `$ |> ...` pipeline, `weights.grad(objective)` — but trained and run on the **MPS backend**, PyTorch's Apple-Silicon GPU device:

* `weights.grad(objective, backend="mps", jit=True)` differentiates the objective with PyTorch's autodiff on the GPU, compiling that gradient **once**.
* `compile_to(forward, "mps")` runs the same `forward` over tensors that live on the Metal device.

MPS has no float64, so this backend computes pycograd's float64 default **in float32**; gradients match the numpy tape to ~1e-6. The setup cell falls back to CPU `torch` when no MPS device is present, so this runs anywhere.

In [1]:
%load_ext pycograd

## Setup

The loss and activations are just numpy used as pipe stages — there is no op library to import.

In [2]:
import logging
import numpy as np
from pycograd import compile_to, get_backend, frozen

def softmax_ce(logits, onehot):
    z = logits - np.max(logits, axis=1, keepdims=True)          # stabilize
    logp = z - np.log(np.sum(np.exp(z), axis=1, keepdims=True)) # log-softmax
    return -np.mean(np.sum(onehot * logp, axis=1))

def relu(z): return np.maximum(0.0, z)

# data: three Gaussian blobs, 3-way classification
rng = np.random.default_rng(0)
m, centers = 40, np.array([[2.0, 2.0], [-2.0, 2.0], [0.0, -2.5]])
X = np.vstack([rng.normal(c, 0.5, (m, 2)) for c in centers])
labels = np.repeat(np.arange(3), m)
Y = np.eye(3)[labels]

# pick the GPU (MPS) backend, falling back to CPU torch when unavailable
import torch
logging.getLogger("torch._inductor.utils").setLevel(logging.ERROR)  # quiet a benign GPU perf note
BACKEND = "mps" if torch.backends.mps.is_available() else "torch"
print("training on backend:", repr(BACKEND))

def train(weights, objective, steps, lr):
    first = last = None
    for _ in range(steps):
        value, grads = weights.grad(objective, backend=BACKEND, jit=True)  # gradient from PyTorch, compiled once
        first = float(value) if first is None else first
        last = float(value)
        weights.step(grads, lr)
    return first, last

def accuracy(logits):
    return float(np.mean(np.argmax(np.asarray(logits), axis=1) == labels))

training on backend: 'mps'


## Train an MLP on the GPU

The forward is one `|>` pipeline; `relu` is just a stage. Same DSL as the numpy demo — only the gradient's source changed.

In [3]:
with params{
    w1 = 0.3 * rng.standard_normal((2, 16))
    b1 = np.zeros(16)
    w2 = 0.3 * rng.standard_normal((16, 3))
    b2 = np.zeros(3)
} as weights:
    forward = $ |> $ @ w1 + b1 |> relu |> $ @ w2 + b2
    objective = lambda: forward(X) |> softmax_ce($, Y)
    first, last = train(weights, objective, 300, 0.5)
    logits = forward(X)

print("loss %.4f -> %.4f" % (first, last))
print("train accuracy:", accuracy(logits))

loss 1.5142 -> 0.0007
train accuracy: 1.0


## Gradients match the numpy tape

`weights.grad(objective)` with no backend runs pycograd's numpy tape; with `backend=BACKEND` it comes from PyTorch on the GPU. They agree to float32 tolerance — and since the tape is finite-difference-checked, that validates the GPU path. `compile_to` runs the same `forward` over device tensors.

In [4]:
with params{
    w1 = 0.3 * rng.standard_normal((2, 16))
    b1 = np.zeros(16)
    w2 = 0.3 * rng.standard_normal((16, 3))
    b2 = np.zeros(3)
} as weights:
    forward = $ |> $ @ w1 + b1 |> relu |> $ @ w2 + b2
    objective = lambda: forward(X) |> softmax_ce($, Y)
    v_np,  g_np  = weights.grad(objective)                  # numpy tape
    v_gpu, g_gpu = weights.grad(objective, backend=BACKEND) # PyTorch autodiff on the GPU
    worst = max(np.max(np.abs(np.asarray(g_gpu[k]) - np.asarray(g_np[k]))) for k in weights)

    out = compile_to(forward, BACKEND)(get_backend(BACKEND).const(X))  # forward on device tensors

print("value  numpy=%.6f  %s=%.6f" % (float(v_np), BACKEND, float(v_gpu)))
print("max |grad_%s - grad_numpy|: %.1e" % (BACKEND, worst))
print("compile_to ran on device:", out.device.type)

value  numpy=1.883657  mps=1.883657
max |grad_mps - grad_numpy|: 5.4e-08
compile_to ran on device: mps


## A frozen parameter

`frozen[...]` holds a weight fixed: its gradient comes back `None` and `weights.step` skips it — same semantics as the numpy tape.

In [5]:
with params{
    w1 = 0.3 * rng.standard_normal((2, 16))
    b1 = frozen[np.zeros(16)]                       # held fixed: gradient comes back None
    w2 = 0.3 * rng.standard_normal((16, 3))
    b2 = np.zeros(3)
} as weights:
    forward = $ |> $ @ w1 + b1 |> relu |> $ @ w2 + b2
    objective = lambda: forward(X) |> softmax_ce($, Y)
    _, g_gpu = weights.grad(objective, backend=BACKEND)
    train(weights, objective, 300, 0.5)

print("frozen b1 gradient:", g_gpu["b1"], " | b1 stayed all-zero:", bool(np.all(weights["b1"].value == 0)))

frozen b1 gradient: None  | b1 stayed all-zero: True


## More

Richer models — highway nets, self-attention, a Transformer block — are written in this DSL in [`pycograd_demo`](pycograd_demo.ipynb), and exporting a trained net to a standalone `nn.Module` / TorchScript / ONNX is shown in [`pycograd_compile_torch_demo`](pycograd_compile_torch_demo.ipynb). The MPS conformance suite checks gradient parity vs the numpy tape:

```
python -m pytest test/test_mps.py
```